Install dependencies

In [1]:
!pip install torch torchvision torchaudio -q
!pip install torch-geometric -q
!pip install gensim -q
!pip install scikit-learn -q
!pip install seaborn -q
!pip install networkx -q
!pip install pandas -q

Imports + GPU Setup

In [2]:
import os
import random
import time
import tracemalloc

import numpy as np
import pandas as pd
import networkx as nx

import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F

from gensim.models import Word2Vec

from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE

from torch_geometric.datasets import Planetoid
from torch_geometric.datasets import CitationFull
import torch_geometric.utils as pyg_utils

sns.set_style("whitegrid")
sns.set_context("paper", font_scale=1.5)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", DEVICE)

Using device: cuda


Utility Functions

In [3]:
def measure_time_memory(func, *args, **kwargs):

    tracemalloc.start()

    start = time.time()

    result = func(*args, **kwargs)

    current, peak = tracemalloc.get_traced_memory()

    tracemalloc.stop()

    return result, time.time() - start, peak / 1e6

Random Walks

In [4]:
def generate_random_walks(G, num_walks=5, walk_length=20):

    walks = []
    nodes = list(G.nodes())

    for _ in range(num_walks):
        random.shuffle(nodes)

        for node in nodes:
            walk = [str(node)]
            current = node

            for _ in range(walk_length - 1):
                neighbors = list(G.neighbors(current))
                if not neighbors:
                    break
                current = random.choice(neighbors)
                walk.append(str(current))

            walks.append(walk)

    return walks

DeepWalk

In [5]:
def train_deepwalk(G, emb_dim=128, epochs=5):

    walks = generate_random_walks(G)

    model = Word2Vec(
        walks,
        vector_size=emb_dim,
        window=5,
        min_count=0,
        sg=1,
        workers=4,
        epochs=epochs
    )

    num_nodes = G.number_of_nodes()

    embeddings = np.array([
        model.wv[str(i)] for i in range(num_nodes)
    ])

    return embeddings

CP-LSH Hashing

In [6]:
def compute_lsh(features, m=6, k=10):

    num_nodes, dim = features.shape

    random_vectors = np.random.randn(dim, m * k)

    random_vectors /= np.linalg.norm(random_vectors, axis=0)

    hashes = np.zeros((num_nodes, m), dtype=np.int64)

    for i in range(num_nodes):

        for h in range(m):

            bucket = 0

            for bit in range(k):

                idx = h * k + bit

                dot = np.dot(features[i], random_vectors[:, idx])

                if dot > 0:
                    bucket |= (1 << bit)

            bucket_id = h * (2 ** k) + bucket

            hashes[i, h] = bucket_id

    return hashes

CP-LSH Model

In [7]:
class CPLSH(nn.Module):

    def __init__(self, total_buckets, emb_dim):

        super().__init__()

        self.src_emb = nn.Embedding(total_buckets, emb_dim)

        self.tgt_emb = nn.Embedding(total_buckets, emb_dim)

        nn.init.xavier_uniform_(self.src_emb.weight)

        nn.init.xavier_uniform_(self.tgt_emb.weight)

    def forward(self, hashes):

        src = self.src_emb(hashes).mean(dim=1)

        tgt = self.tgt_emb(hashes).mean(dim=1)

        return src, tgt

SkipGram Pairs

In [8]:
def generate_skipgram_pairs(walks, window_size=5):

    pairs = []

    for walk in walks:

        walk = [int(x) for x in walk]

        for i, u in enumerate(walk):

            left = max(0, i - window_size)

            right = min(len(walk), i + window_size + 1)

            for j in range(left, right):

                if i == j:
                    continue

                v = walk[j]

                pairs.append((u, v))

    return pairs

Train CP-LSH

In [9]:
def train_cplsh(
    G,
    features,
    emb_dim=128,
    m=8,
    k=9,
    epochs=12,
    neg_samples=10,
    batch_size=512
):

    num_nodes = G.number_of_nodes()

    # ---------- LSH ----------
    hashes = compute_lsh(features, m, k)
    hashes = torch.tensor(hashes).long().to(DEVICE)

    total_buckets = m * (2 ** k)

    model = CPLSH(total_buckets, emb_dim).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    # ---------- Generate SkipGram pairs ----------
    walks = generate_random_walks(G)
    pairs = generate_skipgram_pairs(walks)

    print("Total training pairs:", len(pairs))

    # Convert to tensor once
    pairs = torch.tensor(pairs).long()

    # ---------- Training ----------
    for epoch in range(epochs):

        permutation = torch.randperm(len(pairs))
        pairs = pairs[permutation]

        total_loss = 0

        for i in range(0, len(pairs), batch_size):

            batch = pairs[i:i+batch_size]
            u = batch[:,0].to(DEVICE)
            v = batch[:,1].to(DEVICE)

            optimizer.zero_grad()

            # --- Only compute embeddings for needed nodes ---
            src_u = model.src_emb(hashes[u]).mean(dim=1)
            tgt_v = model.tgt_emb(hashes[v]).mean(dim=1)

            pos_score = torch.sum(src_u * tgt_v, dim=1)
            pos_loss = -F.logsigmoid(pos_score).mean()

            # ---- Negative Sampling ----
            neg_nodes = torch.randint(
                0,
                num_nodes,
                (len(u), neg_samples),
                device=DEVICE
            )

            neg_emb = model.tgt_emb(hashes[neg_nodes]).mean(dim=2)

            src_expand = src_u.unsqueeze(1)

            neg_score = torch.sum(src_expand * neg_emb, dim=2)

            neg_loss = -F.logsigmoid(-neg_score).mean()

            loss = pos_loss + neg_loss

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")

    model.eval()

    with torch.no_grad():
      skipgram_emb = model.src_emb(hashes).mean(dim=1).cpu().numpy()
      content_emb = features
      final_emb = np.concatenate([content_emb, skipgram_emb], axis=1)

    return final_emb

Node Classification

In [10]:
from sklearn.model_selection import StratifiedKFold

def node_classification(embeddings, labels):

    ratios = [0.1, 0.2, 0.3, 0.4, 0.5]
    results = []

    for ratio in ratios:

        skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
        fold_scores = []

        for train_idx, test_idx in skf.split(embeddings, labels):

            train_size = int(len(train_idx) * ratio)
            train_subset = train_idx[:train_size]

            clf = LinearSVC(max_iter=5000)

            clf.fit(embeddings[train_subset], labels[train_subset])

            preds = clf.predict(embeddings[test_idx])

            f1 = f1_score(labels[test_idx], preds, average="micro")

            fold_scores.append(f1)

        results.append((ratio, np.mean(fold_scores)))

    return results

Tail vs Head Analysis

In [11]:
def tail_head_analysis(G, embeddings, labels):

    degrees = dict(G.degree())

    threshold = np.percentile(list(degrees.values()), 20)

    tail_nodes = [i for i,d in degrees.items() if d <= threshold]

    head_nodes = [i for i,d in degrees.items() if d > threshold]

    idx = np.arange(len(labels))

    train_idx, test_idx = train_test_split(
        idx,
        test_size=0.5,
        stratify=labels
    )

    clf = LinearSVC(max_iter=5000)

    clf.fit(embeddings[train_idx], labels[train_idx])

    preds = clf.predict(embeddings[test_idx])

    true = labels[test_idx]

    tail_mask = [i for i,x in enumerate(test_idx) if x in tail_nodes]
    head_mask = [i for i,x in enumerate(test_idx) if x in head_nodes]

    tail_f1 = f1_score(
        true[tail_mask],
        preds[tail_mask],
        average="micro"
    )

    head_f1 = f1_score(
        true[head_mask],
        preds[head_mask],
        average="micro"
    )

    return tail_f1, head_f1

Link Prediction

In [12]:
def link_prediction_curve(G, embeddings, ratios=None):

    if ratios is None:
        ratios = np.arange(0.1, 1.0, 0.1)

    edges = list(G.edges())
    nodes = list(G.nodes())

    auc_scores = []

    for ratio in ratios:

        np.random.shuffle(edges)
        num_train = int(ratio * len(edges))

        train_edges = edges[:num_train]
        test_edges = edges[num_train:]

        neg_edges = []
        while len(neg_edges) < len(test_edges):
            u, v = np.random.choice(nodes, 2, replace=False)
            if not G.has_edge(u, v):
                neg_edges.append((u, v))

        y_true = [1]*len(test_edges) + [0]*len(neg_edges)
        y_scores = []

        for u, v in test_edges:
            y_scores.append(np.dot(embeddings[u], embeddings[v]))

        for u, v in neg_edges:
            y_scores.append(np.dot(embeddings[u], embeddings[v]))

        auc = roc_auc_score(y_true, y_scores)
        auc_scores.append(auc)

    return ratios, auc_scores

Degree Distribution Plot

In [13]:
def plot_degree_distribution(G, dataset_name):

    degrees = [d for n,d in G.degree()]

    plt.figure(figsize=(6,4))

    plt.hist(degrees, bins=50)

    plt.yscale("log")

    plt.xlabel("Degree")

    plt.ylabel("Frequency")

    plt.title(f"{dataset_name} Degree Distribution")

    plt.tight_layout()

    plt.savefig(f"{dataset_name}_degree_distribution.png")

    plt.close()

t-SNE Visualization

In [14]:
def tsne_plot(embeddings, labels, dataset_name, method_name):

    tsne = TSNE(
      n_components=2,
      perplexity=30,
      n_iter=500,
      random_state=42
    )

    emb_2d = tsne.fit_transform(embeddings)

    plt.figure(figsize=(7,6))

    plt.scatter(
        emb_2d[:,0],
        emb_2d[:,1],
        c=labels,
        cmap="tab10",
        s=8
    )

    plt.title(f"{dataset_name} - {method_name} tSNE")

    plt.tight_layout()

    plt.savefig(f"{dataset_name}_{method_name}_tsne.png")

    plt.close()

Sensitivity Heatmap

In [15]:
def sensitivity_analysis(G, features, labels, dataset_name):

    m_values = [4,6,8,10]

    k_values = [6,8,10,12]

    results = []

    for m in m_values:

        for k in k_values:

            print(f"Running m={m}, k={k}")

            embs = train_cplsh(
                G,
                features,
                m=m,
                k=k,
                epochs=3
            )

            scores = node_classification(embs, labels)

            avg_f1 = np.mean([x[1] for x in scores])

            results.append((m,k,avg_f1))

    df = pd.DataFrame(results, columns=["m","k","F1"])

    pivot = df.pivot(index="m", columns="k", values="F1")

    plt.figure(figsize=(8,6))

    sns.heatmap(
        pivot,
        annot=True,
        cmap="viridis"
    )

    plt.title(f"{dataset_name} Sensitivity Heatmap")

    plt.tight_layout()

    plt.savefig(f"{dataset_name}_sensitivity_heatmap.png")

    plt.close()

In [16]:
def train_svd(features, emb_dim=80):
    from sklearn.decomposition import TruncatedSVD

    svd = TruncatedSVD(n_components=emb_dim, random_state=42)
    return svd.fit_transform(features)

Main Experiment

In [17]:
import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = "results"
if not os.path.exists(RESULTS_DIR):
    os.makedirs(RESULTS_DIR)

datasets = ["Cora", "Citeseer", "DBLP"]

all_results = []

for dataset_name in datasets:

    print(f"\n==============================")
    print(f"Running on {dataset_name}")
    print(f"==============================")

    # ===============================
    # Dataset Loading
    # ===============================
    if dataset_name in ["Cora", "Citeseer"]:
        dataset = Planetoid(root=f"./data/{dataset_name}", name=dataset_name)
    else:
        dataset = CitationFull(root="./data/DBLP", name="DBLP")

    data = dataset[0]
    G = pyg_utils.to_networkx(data, to_undirected=True)

    features = data.x.numpy()
    features = features / (np.linalg.norm(features, axis=1, keepdims=True) + 1e-9)
    labels = data.y.numpy()

    # ===============================
    # Train Models
    # ===============================
    print("\nTraining DeepWalk")
    dw_embs, dw_time, dw_mem = measure_time_memory(train_deepwalk, G)

    print("\nTraining SVD")
    svd_embs, svd_time, svd_mem = measure_time_memory(train_svd, features)

    print("\nCreating DW+SVD")
    dw_svd_embs = np.concatenate([dw_embs, svd_embs], axis=1)

    print("\nTraining CP-LSH")
    cp_embs, cp_time, cp_mem = measure_time_memory(train_cplsh, G, features)

    # ===============================
    # Node Classification
    # ===============================
    dw_f1 = node_classification(dw_embs, labels)
    svd_f1 = node_classification(svd_embs, labels)
    dw_svd_f1 = node_classification(dw_svd_embs, labels)
    cp_f1 = node_classification(cp_embs, labels)

    # ===============================
    # Tail vs Head Analysis
    # ===============================
    dw_tail, dw_head = tail_head_analysis(G, dw_embs, labels)
    cp_tail, cp_head = tail_head_analysis(G, cp_embs, labels)

    tail_gap_dw = dw_head - dw_tail
    tail_gap_cp = cp_head - cp_tail

    plt.figure(figsize=(6,5))
    plt.bar(["DW Tail","DW Head","CP Tail","CP Head"],
            [dw_tail, dw_head, cp_tail, cp_head])
    plt.ylabel("Micro-F1")
    plt.title(f"{dataset_name} Tail vs Head")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f"{dataset_name}_tail_head.png"))
    plt.close()

    # ===============================
    # Link Prediction Curve
    # ===============================
    ratios, dw_auc_curve = link_prediction_curve(G, dw_embs)
    _, svd_auc_curve = link_prediction_curve(G, svd_embs)
    _, dw_svd_auc_curve = link_prediction_curve(G, dw_svd_embs)
    _, cp_auc_curve = link_prediction_curve(G, cp_embs)

    plt.figure(figsize=(7,5))
    plt.plot(ratios*100, dw_auc_curve, marker='o', label="DeepWalk")
    plt.plot(ratios*100, svd_auc_curve, marker='o', label="SVD")
    plt.plot(ratios*100, dw_svd_auc_curve, marker='o', label="DW+SVD")
    plt.plot(ratios*100, cp_auc_curve, marker='o', label="CP-LSH")

    plt.xlabel("Training Edges (%)")
    plt.ylabel("AUC")
    plt.title(f"{dataset_name} Link Prediction Curve")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f"{dataset_name}_link_prediction_curve.png"))
    plt.close()

    # ===============================
    # Classification Curve
    # ===============================
    plt.figure(figsize=(7,5))
    plt.plot([x[0]*100 for x in dw_f1], [x[1] for x in dw_f1], marker='o', label="DeepWalk")
    plt.plot([x[0]*100 for x in svd_f1], [x[1] for x in svd_f1], marker='o', label="SVD")
    plt.plot([x[0]*100 for x in dw_svd_f1], [x[1] for x in dw_svd_f1], marker='o', label="DW+SVD")
    plt.plot([x[0]*100 for x in cp_f1], [x[1] for x in cp_f1], marker='o', label="CP-LSH")

    plt.xlabel("% Training Nodes")
    plt.ylabel("Micro-F1")
    plt.title(f"{dataset_name} Classification")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f"{dataset_name}_classification_curve.png"))
    plt.close()

    # ===============================
    # Runtime + Memory
    # ===============================
    plt.figure(figsize=(6,5))
    plt.bar(["DW","SVD","CP"], [dw_time, svd_time, cp_time])
    plt.ylabel("Seconds")
    plt.title(f"{dataset_name} Runtime")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f"{dataset_name}_runtime.png"))
    plt.close()

    plt.figure(figsize=(6,5))
    plt.bar(["DW","SVD","CP"], [dw_mem, svd_mem, cp_mem])
    plt.ylabel("MB")
    plt.title(f"{dataset_name} Memory")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f"{dataset_name}_memory.png"))
    plt.close()

    # ===============================
    # Save Results Row
    # ===============================
    all_results.append([
        dataset_name,
        dw_time, dw_mem, np.mean(dw_auc_curve),
        cp_time, cp_mem, np.mean(cp_auc_curve),
        tail_gap_dw, tail_gap_cp
    ])

# ===============================
# Save Final CSV
# ===============================
results_df = pd.DataFrame(
    all_results,
    columns=[
        "Dataset",
        "DW_Time","DW_Memory","DW_AUC_mean",
        "CP_Time","CP_Memory","CP_AUC_mean",
        "DW_Tail_Gap","CP_Tail_Gap"
    ]
)

results_df.to_csv(os.path.join(RESULTS_DIR, "final_results.csv"), index=False)

# ===============================
# Zip Everything
# ===============================
shutil.make_archive("experiment_outputs", 'zip', RESULTS_DIR)

print("\nALL EXPERIMENTS FINISHED")
print("ZIP file created: experiment_outputs.zip")


Running on Cora

Training DeepWalk

Training SVD

Creating DW+SVD

Training CP-LSH
Total training pairs: 2301800
Epoch 1 Loss: 2676.7547
Epoch 2 Loss: 1384.6208
Epoch 3 Loss: 1204.3077
Epoch 4 Loss: 1123.0890
Epoch 5 Loss: 1073.6548
Epoch 6 Loss: 1039.4789
Epoch 7 Loss: 1013.2534
Epoch 8 Loss: 992.8462
Epoch 9 Loss: 974.2425
Epoch 10 Loss: 961.1550
Epoch 11 Loss: 946.7884
Epoch 12 Loss: 935.9577

Running on Citeseer

Training DeepWalk

Training SVD

Creating DW+SVD

Training CP-LSH
Total training pairs: 2787150
Epoch 1 Loss: 1964.5973
Epoch 2 Loss: 637.6450
Epoch 3 Loss: 544.0085
Epoch 4 Loss: 506.7938
Epoch 5 Loss: 486.5215
Epoch 6 Loss: 472.0308
Epoch 7 Loss: 461.5410
Epoch 8 Loss: 453.6950
Epoch 9 Loss: 446.3090
Epoch 10 Loss: 441.1568
Epoch 11 Loss: 437.0998
Epoch 12 Loss: 433.0731

Running on DBLP

Training DeepWalk

Training SVD

Creating DW+SVD

Training CP-LSH
Total training pairs: 15058600
Epoch 1 Loss: 24320.1902
Epoch 2 Loss: 18514.7919
Epoch 3 Loss: 17272.0107
Epoch 4 Loss